# ME324 · Lab 11 — Text-gen 4/4: tokenisation & sampling

**Lecture 11 · "Building your own text-generating AI (4/4)" · 2026-08-18**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/labs/lab-11-bpe-sampling-capstone.ipynb)

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

---

### Where we are

The last build lab of the course. Yesterday you built a working GPT; today it gets a
smarter **tokeniser** (byte-pair encoding) and smarter **sampling**. Then we can build
a GPT in *pure Python*, no PyTorch at all, running entirely on the
`Value` autograd class you wrote in Labs 3-4!

### Goals

1. **BPE**: implement `merge`, `train_bpe` and `encode`; measure the compression on
   tiny-shakespeare.
2. **Sampling controls**: temperature, top-k, top-p — the quality/diversity trade-off.
3. **The capstone**: run Karpathy's pure-Python microGPT and *see* that your
   transformer is "just" `Value` plus loops.

## ⏱️ Plan for today (~90 minutes)

Built for a single 90-minute session — it's **completely fine not to finish every cell in the room**.

- **Core — do these:** the BPE trio (`merge`, `train_bpe`, `encode`), the sampler in Section 2, and **running the capstone** — run-and-watch, not implement, so keep ten minutes back for it.
- **Stretch / take-home — skip if short on time:** the token-boundary and greedy-vs-sampled comparisons (1.9 and 2.4).

_The `# TODO` cells are the parts you write; worked answers are in the **Solutions** section at the bottom._

## Run me first

This cell imports what we need, downloads **tiny-shakespeare**, and fixes the random
seed so your run matches the notes.

> **Tip:** set *Runtime → Change runtime type → GPU* in Colab; on CPU, lower
> `max_iters` below.

In [ ]:
import math, random, urllib.request, os
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
random.seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Download tiny-shakespeare to shakespeare.txt
# (We deliberately do NOT name it input.txt: the capstone in Section 3 downloads its
#  own input.txt of names, and we want to keep that gist byte-for-byte unchanged.)
SHAKESPEARE_URL = ('https://raw.githubusercontent.com/karpathy/char-rnn/'
                   'master/data/tinyshakespeare/input.txt')
if not os.path.exists('shakespeare.txt'):
    urllib.request.urlretrieve(SHAKESPEARE_URL, 'shakespeare.txt')
text = open('shakespeare.txt').read()
print(f'length of dataset in characters: {len(text):,}')
print(text[:250])

## Section 0 · The GPT you built yesterday

Labs 8-10 built a reusable **text pipeline** (character tokeniser, batching, training
loop, loss estimation) and models sharing one interface: `model(idx, targets)` returns
`(logits, loss)`, and `model.generate` extends a sequence. We re-create both so this lab
stands alone. **None of this is new** — the new material starts in Section 1.

### 0.1 · The character pipeline (Lab 8)

`build_char_pipeline` rebuilds the character ↔ integer maps, the train/val split,
`get_batch`, and `estimate_loss`.

In [ ]:
def build_char_pipeline(text, block_size=32, batch_size=32, device='cpu', seed=1337):
    torch.manual_seed(seed)
    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join(itos[i] for i in l)
    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    train_data, val_data = data[:n], data[n:]

    def get_batch(split):
        d = train_data if split == 'train' else val_data
        ix = torch.randint(len(d) - block_size, (batch_size,))
        x = torch.stack([d[i:i + block_size] for i in ix])
        y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
        return x.to(device), y.to(device)

    @torch.no_grad()
    def estimate_loss(model, eval_iters=200):
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                xb, yb = get_batch(split)
                _, loss = model(xb, yb)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    return dict(vocab_size=vocab_size, stoi=stoi, itos=itos, encode=encode,
                decode=decode, get_batch=get_batch, estimate_loss=estimate_loss,
                block_size=block_size, device=device)

BLOCK_SIZE = 32
P = build_char_pipeline(text, block_size=BLOCK_SIZE, batch_size=32, device=device)
print('char-level vocab size:', P['vocab_size'])
print('first 32 chars encoded:', P['encode'](text[:32]))

### 0.2 · The GPT (Lab 10)

The full transformer from yesterday, wrapped in a factory. Its `generate` already
handles `temperature` and `top_k`; Section 2 adds **top-p**.

In [ ]:
def make_gpt_model(block_size, vocab_size, n_embd=64, n_head=4, n_layer=4, dropout=0.0):
    class Head(nn.Module):
        def __init__(self, head_size):
            super().__init__()
            self.key = nn.Linear(n_embd, head_size, bias=False)
            self.query = nn.Linear(n_embd, head_size, bias=False)
            self.value = nn.Linear(n_embd, head_size, bias=False)
            self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
            self.dropout = nn.Dropout(dropout)
        def forward(self, x):
            B, T, C = x.shape
            k = self.key(x); q = self.query(x)
            wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            v = self.value(x)
            return wei @ v

    class MultiHeadAttention(nn.Module):
        def __init__(self, num_heads, head_size):
            super().__init__()
            self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
            self.proj = nn.Linear(head_size * num_heads, n_embd)
            self.dropout = nn.Dropout(dropout)
        def forward(self, x):
            out = torch.cat([h(x) for h in self.heads], dim=-1)
            return self.dropout(self.proj(out))

    class FeedForward(nn.Module):
        def __init__(self, n_embd):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
                nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
        def forward(self, x): return self.net(x)

    class Block(nn.Module):
        def __init__(self, n_embd, n_head):
            super().__init__()
            head_size = n_embd // n_head
            self.sa = MultiHeadAttention(n_head, head_size)
            self.ffwd = FeedForward(n_embd)
            self.ln1 = nn.LayerNorm(n_embd)
            self.ln2 = nn.LayerNorm(n_embd)
        def forward(self, x):
            x = x + self.sa(self.ln1(x))
            x = x + self.ffwd(self.ln2(x))
            return x

    class GPTLanguageModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
            self.position_embedding_table = nn.Embedding(block_size, n_embd)
            self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(n_embd)
            self.lm_head = nn.Linear(n_embd, vocab_size)
        def forward(self, idx, targets=None):
            B, T = idx.shape
            tok_emb = self.token_embedding_table(idx)
            pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
            x = tok_emb + pos_emb
            x = self.blocks(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                B, T, C = logits.shape
                loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
            return logits, loss
        @torch.no_grad()
        def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
            for _ in range(max_new_tokens):
                idx_cond = idx[:, -block_size:]
                logits, _ = self(idx_cond)
                logits = logits[:, -1, :] / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('inf')
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)
                idx = torch.cat((idx, idx_next), dim=1)
            return idx
    return GPTLanguageModel

### 0.3 · Train it briefly

A couple of minutes of training gives us a model worth sampling from. Raise `max_iters`
for nicer output; lower it on CPU.

In [ ]:
GPT = make_gpt_model(block_size=BLOCK_SIZE, vocab_size=P['vocab_size'],
                     n_embd=64, n_head=4, n_layer=4)
gpt = GPT().to(device)
print(sum(p.numel() for p in gpt.parameters()), 'parameters')

optimizer = torch.optim.AdamW(gpt.parameters(), lr=1e-3)
max_iters, eval_interval = 3000, 500      # lower these if you are on CPU
for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = P['estimate_loss'](gpt)
        print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
    xb, yb = P['get_batch']('train')
    _, loss = gpt(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

**This is the GPT you built yesterday.**

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
out = gpt.generate(context, max_new_tokens=300)
print(P['decode'](out[0].tolist()))

## Section 1 · Byte-pair encoding (BPE)

A character tokeniser keeps the vocabulary tiny (~65) but sequences long — one token
per letter — so the model spends capacity learning spelling. Real LLMs use **subword**
vocabularies of ~32k-128k pieces: no out-of-vocabulary words, far shorter sequences.
The standard algorithm, **byte-pair encoding (BPE)**, from Lecture 11:

> Start with every byte as a token (vocab = 256). Then, until the target vocab size:
> **(1)** count adjacent token *pairs*, **(2)** find the most frequent pair,
> **(3)** merge it into a new token, **(4)** add it to the vocabulary.

As we said in the lecture, this is compression tech from the 1990s; the default in every 
major LLM since Sennrich et al. (2016).

### 1.1 · BPE by hand (Lecture 11 plenary)

The plenary corpus — `"banana banana banana ana ana"`, with `_` marking word-ends and
each character a token:

| Step | Merge | After |
|---|---|---|
| 0 | *(start)* | `b a n a n a _`×3 · `a n a _`×2 |
| 1 | `(n,a)→na` | `b a na na _`×3 · `a na _`×2 |
| 2 | `(a,na)→ana` | `b ana na _`×3 · `ana _`×2 |
| 3 | `(b,ana)→bana` | `bana na _`×3 · `ana _`×2 |

Three merges make **`banana`** almost one token, and the never-seen **`bandana`** still
encodes as `b a n d ana _` — known pieces only: **no word is ever out-of-vocabulary.**
Our code works on **raw utf-8 bytes** (like GPT-2), so its merges will differ slightly;
same algorithm.

### 1.2 · `get_stats` — count adjacent pairs

Step (1): count how often each adjacent pair occurs:

In [ ]:
def get_stats(ids):
    """Count adjacent pairs. ids: list[int]. Returns {(a,b): count}."""
    counts = {}
    for a, b in zip(ids, ids[1:]):
        counts[(a, b)] = counts.get((a, b), 0) + 1
    return counts

# quick check on the bytes of a tiny string
demo_ids = list("aaabdaaabac".encode("utf-8"))
print("ids:", demo_ids)
print("pair counts:", get_stats(demo_ids))

### 1.3 · `merge` — replace a pair with a new id

Step (3): wherever `pair` occurs back-to-back in `ids`, emit the new token `idx`
instead. The full spec is in the docstring.

In [ ]:
def merge(ids, pair, idx):
    """Replace each adjacent occurrence of `pair` in `ids` with the single token `idx`.

    Spec:
      - `ids`  : list[int] of current token ids.
      - `pair` : tuple (a, b) of two token ids to fuse.
      - `idx`  : the new token id to emit in their place.
      - Return a NEW list where every time `a` is immediately followed by `b`,
        the two are replaced by a single `idx`. Non-matching tokens are copied as-is.
        Walk left to right; after a match, skip BOTH tokens of the pair.
      - Example: merge([97,97,98,97,97], (97,97), 256) -> [256, 98, 256]
    """
    # TODO: implement merge (see Solutions section if stuck)
    raise NotImplementedError

# merge the pair (a=97,a=97) -> new token 256 in our demo
print(merge(demo_ids, (97, 97), 256))

### 1.4 · `train_bpe` — learn the merges

Steps (1)-(4) in a loop: repeatedly find the most frequent pair, merge it, record it —
`vocab_size - 256` times, since every possible byte is already a token.

In [ ]:
def train_bpe(text, vocab_size, base=256, verbose=False):
    """Learn BPE merges from `text`. Returns {pair: new_id} in the order learned.

    Spec:
      - Encode `text` to a list of utf-8 byte ids (0..255) -> `ids`.
      - Repeat `vocab_size - base` times:
          (1) count adjacent pairs with get_stats(ids);
          (2) if there are no pairs left, stop;
          (3) pick the MOST FREQUENT pair;
          (4) give it the next new id (base, base+1, ...), merge() it in `ids`,
              and record it in `merges`.
      - Return `merges`, a dict {pair: new_id} IN THE ORDER LEARNED
        (insertion order matters for encode()).
    """
    # TODO: implement train_bpe (see Solutions section if stuck)
    raise NotImplementedError

### 1.5 · `encode` / `decode`

`encode` applies the learned merges to *new* text **in the order they were learned** —
later merges build on earlier ones. Complete the loop then read through `decode`, which
expands tokens back into bytes.

In [ ]:
def encode(text, merges):
    """Tokenise `text` using learned `merges` (apply earliest-learned merge first)."""
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # TODO 1: of all adjacent pairs in `stats`, pick the one whose id in `merges`
        #   is LOWEST (i.e. learned earliest).
        #   Two things to get right: min() accepts a key function, and a pair we
        #   never learned must never win (an infinite score does it).
        pair = ...                                   # <-- fill this in
        if pair not in merges:
            break                      # nothing left that we know how to merge
        # TODO 2: apply that merge to `ids` -- one call to a function you wrote above.
        ids = ...                                    # <-- fill this in
    return ids

def decode(ids, merges):
    """Turn token ids back into a string."""
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), idx in merges.items():
        vocab[idx] = vocab[a] + vocab[b]
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

### 1.6 · Sanity check on the plenary corpus

On raw bytes you won't match the hand table exactly, but frequent pieces fuse all the same.

In [ ]:
banana = "banana banana banana ana ana"
m_banana = train_bpe(banana, vocab_size=256 + 5, verbose=True)
print("learned merges:", m_banana)
enc = encode(banana, m_banana)
print("encoded length:", len(enc), "(was", len(banana.encode('utf-8')), "bytes)")
print("round-trips:", decode(enc, m_banana) == banana)

In [ ]:
# Notice that "bandana" never appears in the corpus,
# yet the learned merges still build it out of known pieces.
# TODO 1: encode "bandana" with the merges learned above (m_banana)
band_ids = None    # <-- TODO
# TODO 2: decode the ids back to a string
band_back = None   # <-- TODO

if band_ids is None:
    print("Fill in the TODOs above, then run this cell again.")
else:
    print("token ids  :", band_ids, f"({len(band_ids)} tokens for 7 bytes)")
    print("round-trips:", band_back == "bandana")

### 1.7 · Train BPE on tiny-shakespeare

Pure-Python BPE re-scans the sequence on every merge, so we train on a **slice** with a
modest number of merges; real tokenisers use optimised code on the full corpus.

In [ ]:
bpe_train_text = text[:20000]          # a slice, to keep pure-Python BPE fast
TARGET_VOCAB = 512                      # 256 bytes + 256 learned merges
merges = train_bpe(bpe_train_text, vocab_size=TARGET_VOCAB)
print(f"learned {len(merges)} merges; BPE vocab size = {256 + len(merges)}")

# Peek at some merged tokens: what byte-strings did the new ids come to mean?
vocab = {i: bytes([i]) for i in range(256)}
for (a, b), idx in merges.items():
    vocab[idx] = vocab[a] + vocab[b]
some = [256, 257, 258, 300, 350, 400, 450, 500]
for i in some:
    if i in vocab:
        print(f"token {i!r:>4} -> {vocab[i]!r}")

### 1.8 · Char-level vs BPE: sequence length

Because attention cost scales with the *square* of sequence length,
shorter sequences are cheaper. Let's measure it on held-out text:

In [ ]:
sample = text[20000:20600]      # held-out text the tokeniser was not trained on

# TODO 1: encode `sample` with the Lab-8 character tokeniser (it lives in P)
char_ids = None     # <-- TODO
# TODO 2: encode `sample` with your BPE and the tiny-shakespeare `merges`
bpe_ids = None      # <-- TODO
# TODO 3: how many times fewer tokens is that? (char tokens per BPE token)
compression = None  # <-- TODO

if char_ids is None:
    print("Fill in the TODOs above, then run this cell again.")
else:
    print(f"characters : {len(sample)}")
    print(f"char tokens: {len(char_ids)}")
    print(f"BPE tokens : {len(bpe_ids)}")
    print(f"compression: {compression:.2f}x fewer tokens with BPE")
    print("round-trips:", decode(bpe_ids, merges) == sample)

### 1.9 · A token-boundary example

Tokenisation decides where the model's "atoms" are — here are the pieces BPE actually
chose for a short phrase.

In [ ]:
# TODO: before you run this cell, predict: how many BPE tokens will the phrase
# become (it is 15 characters), and does 'the' survive as a single piece?
# Then run it, and write down what you find in the comment at the bottom.
phrase = "the king speaks"
ids = encode(phrase, merges)
pieces = [repr(vocab[i].decode('utf-8', errors='replace')) for i in ids]
print(f"{phrase!r}")
print(f"  char tokeniser: {len(P['encode'](phrase))} tokens")
print(f"  BPE tokeniser : {len(ids)} tokens ->", " | ".join(pieces))

# Your answer -- which chunks fused, and why those?
#

> **Optional (slow):** you *could* re-tokenise the corpus with BPE and retrain the
> GPT — only `vocab_size` changes. We leave it optional: today's point is understanding
> tokenisation, not squeezing out a better model.

## Section 2 · Sampling controls — temperature, top-k, top-p

The model outputs **logits**; softmax turns them into a distribution over the next
token. *How* we pick from it changes the output's character (Lecture 11):

- **Temperature** $\tau$: divide logits by $\tau$ before softmax,
  $p_i \propto \exp(z_i/\tau)$ — $\tau < 1$ **sharpens** (confident, repetitive),
  $\tau > 1$ **flattens** (creative, chaotic).
- **Top-k**: keep the $k$ most probable tokens — cuts the garbage tail, but the same
  $k$ however confident the model is.
- **Top-p (nucleus)**: keep the *smallest* set with cumulative probability above $p$ —
  **adapts**: few tokens when the model is sure, many when unsure.

### 2.1 · Implement the sampler

The generation loop is written, so you just need to complete thee three numbered TODOs. 

(Top-p is the fiddly one — read its spec twice before writing anything.)

In [ ]:
@torch.no_grad()
def generate_with_controls(model, idx, max_new_tokens, block_size,
                           temperature=1.0, top_k=None, top_p=None):
    """Autoregressively sample from `model` with temperature / top-k / top-p controls."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]               # crop context to block_size
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]                     # (B, vocab) -- last position only

        # TODO 1: TEMPERATURE -- divide the logits by `temperature`.
        logits = ...                                  # <-- fill this in

        # TODO 2: TOP-K -- keep each row's top_k largest logits; set the rest to -inf.
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            # v[:, [-1]] is the k-th largest logit per row (the cutoff).
            ...                                       # <-- set logits below the cutoff to -float('inf')

        # TODO 3: TOP-P (nucleus). Sort logits descending; softmax -> probs; take the
        #   cumulative sum; set to -inf every token whose cumulative probability BEFORE
        #   it already exceeds top_p (that keeps the smallest set that first crosses
        #   top_p); scatter the filtered logits back to their original positions.
        #   (a handful of lines: torch.sort, cumsum, scatter)
        if top_p is not None:
            ...                                       # <-- fill this in

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

### 2.2 · The same prompt, four ways

Since we use the same seed etc., we can now test the effect of each of these hyperparameters:

In [ ]:
prompt = "ROMEO:"
start = torch.tensor([P['encode'](prompt)], dtype=torch.long, device=device)

def sample(temp, top_k=None, top_p=None, n=200):
    torch.manual_seed(1337)   # same seed -> differences are due to the knobs, not luck
    out = generate_with_controls(gpt, start, n, BLOCK_SIZE,
                                 temperature=temp, top_k=top_k, top_p=top_p)
    return P['decode'](out[0].tolist())

print("=== A. low temperature 0.4 (sharp, safe, repetitive) ===")
print(sample(0.4)); print()
print("=== B. temperature 0.8 + top-k 40 (the workhorse) ===")
print(sample(0.8, top_k=40)); print()
print("=== C. temperature 0.8 + top-p 0.9 (nucleus; modern chat default) ===")
print(sample(0.8, top_p=0.9)); print()
print("=== D. high temperature 1.5, no truncation (wild, often nonsense) ===")
print(sample(1.5))

### 2.3 · What you should notice

The block headers say what to expect; the subtlety is that top-p **adapts** (i.e. few
options when the model is sure, many when not) whereas top-k is fixed. As Lecture 11 put
it, **sampling is interface design**: the same weights yield terse or rambling prose
depending on these configurations.

### 2.4 · Greedy vs sampled

Push the temperature towards zero and softmax collapses onto the single largest logit:
sampling becomes **greedy decoding**, and generation becomes (near-)deterministic.
Worth seeing rather than believing.

In [ ]:
# TODO: generate from `start` TWICE at temperature 0.05 (near-greedy) and TWICE at
# temperature 1.0 -- four runs of 150 tokens each with generate_with_controls,
# WITHOUT re-seeding in between.
# Before you run: which pair should come out (near-)identical, and why?



## Section 3 · The capstone — a GPT in pure Python, on *your* `Value` class

Everything today ran on PyTorch — convenient, fast, a little bit magic. But you built
that magic yourself in **Labs 3-4**. Andrej Karpathy's **microGPT** below is a complete
GPT — training and inference — in **pure, dependency-free Python**: no PyTorch, no
NumPy, every number a `Value` object, **the same class you wrote**, in the architecture
you built in Lab 10 — spelled out in loops over scalars.

| In the gist | You built it in |
|---|---|
| `class Value` (autograd: `+`, `*`, `exp`, `log`, `backward`) | **Labs 3-4** — *the* core of the course |
| `uchars` / `BOS` tokenizer | **Lab 8** — the text pipeline |
| `linear`, `softmax`, `rmsnorm`, attention, MLP, residuals | **Lab 10** — the transformer |
| Adam (moments `m`, `v`, bias-correction, lr decay) | **Lab 4** by hand → `AdamW` (Lab 5+) |
| `temperature` sampling at inference | **Section 2**, just now |

It trains on a list of **names**, then hallucinates new ones. Read it — you will
recognise every piece.

> *Attribution: the next cell is Karpathy's microGPT gist
> (https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95), reproduced
> verbatim except `num_steps` (1000 → 150) so it finishes inside a Colab cell.
> Restore `num_steps = 1000` for a real run.*

In [ ]:
"""
The most atomic way to train and run inference for a GPT in pure, dependency-free Python.
This file is the complete algorithm.
Everything else is just efficiency.

@karpathy
"""

import os       # os.path.exists
import math     # math.log, math.exp
import random   # random.seed, random.choices, random.gauss, random.shuffle
random.seed(42) # Let there be order among chaos

# Let there be a Dataset `docs`: list[str] of documents (e.g. a list of names)
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")

# Let there be a Tokenizer to translate strings to sequences of integers ("tokens") and back
uchars = sorted(set(''.join(docs))) # unique characters in the dataset become token ids 0..n-1
BOS = len(uchars) # token id for a special Beginning of Sequence (BOS) token
vocab_size = len(uchars) + 1 # total number of unique tokens, +1 is for BOS
print(f"vocab size: {vocab_size}")

# Let there be Autograd to recursively apply the chain rule through a computation graph
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads') # Python optimization for memory usage

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # scalar value of this node calculated during forward pass
        self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children       # children of this node in the computation graph
        self._local_grads = local_grads # local derivative of this node w.r.t. its children

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

# Initialize the parameters, to store the knowledge of the model
n_layer = 1     # depth of the transformer neural network (number of layers)
n_embd = 16     # width of the network (embedding dimension)
block_size = 16 # maximum context length of the attention window (note: the longest name is 15 characters)
n_head = 4      # number of attention heads
head_dim = n_embd // n_head # derived dimension of each head
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row] # flatten params into a single list[Value]
print(f"num params: {len(params)}")

# Define the model architecture: a function mapping tokens and parameters to logits over what comes next
# Follow GPT-2, blessed among the GPTs, with minor differences: layernorm -> rmsnorm, no biases, GeLU -> ReLU
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id] # token embedding
    pos_emb = state_dict['wpe'][pos_id] # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
    x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

    for li in range(n_layer):
        # 1) Multi-head Attention block
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]
        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

# Let there be Adam, the blessed optimizer and its buffers
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # first moment buffer
v = [0.0] * len(params) # second moment buffer

# Repeat in sequence
num_steps = 150 # number of training steps  # <-- ONLY change from the original gist: 1000 -> 150 for a live Colab demo
for step in range(num_steps):

    # Take single document, tokenize it, surround it with BOS special token on both sides
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Forward the token sequence through the model, building up the computation graph all the way to the loss
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses) # final average loss over the document sequence. May yours be low.

    # Backward the loss, calculating the gradients with respect to all model parameters
    loss.backward()

    # Adam optimizer update: update the model parameters based on the corresponding gradients
    lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\r')

# Inference: may the model babble back to us
temperature = 0.5 # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")

### Deep learning is magic (in my opinion)

Those names came out of **scalar arithmetic and the chain rule** — your `Value` class,
which we looped a few thousand times. PyTorch makes it *fast* and *parallel* but it adds **nothing
conceptual** that you have not already built by hand.

> At 150 steps the names are rough. Raise `num_steps` (slowly — it's pure Python) and
> they sharpen. *"Everything else is just efficiency."* — Karpathy.

## You built all of this

The arc of ME324, every layer built on the last:

1. **A neuron in NumPy** (Labs 1-2).
2. **Autograd from scratch** (Labs 3-4) — the chain rule as code.
3. **PyTorch** (Lab 5) — the same net, in a framework.
4. **CNNs** (Lab 6) — seeing images.
5. **Autoencoders → VAEs** (Lab 7) — generating them.
6. **A text pipeline + bigram LM** (Lab 8).
7. **RNNs** (Lab 9) — memory across a sequence.
8. **The transformer** (Lab 10) — a real GPT.
9. **BPE + sampling** (today).
10. **A pure-Python GPT on your own autograd** — the whole thing, demystified.

You started with black boxes; **you can now build one from a blank file**. Same skeleton
as GPT-4, Claude, and Gemini — the differences are scale, data, and post-training, not
magic.

### Tomorrow — Lab 12: build them, then critique them

The final lab takes the critic's view: we probe the Lab-8 embeddings for **bias** (a
WEAT-style test) and **red-team** the Lab-10/11 model.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — `merge` (Section 1.3)**

The one subtlety is the step size: after a match you must skip *both* tokens of the
pair, otherwise you would re-match overlapping occurrences.

In [ ]:
def merge(ids, pair, idx):
    """Replace each adjacent occurrence of `pair` in `ids` with the single token `idx`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(idx); i += 2          # matched the pair -> emit new token, skip 2
        else:
            out.append(ids[i]); i += 1        # no match -> copy this token, step 1
    return out

assert merge([97, 97, 98, 97, 97], (97, 97), 256) == [256, 98, 256]
print("merge OK")

**Solution — `train_bpe` (Section 1.4)**

`max(stats, key=stats.get)` returns the pair with the largest count — step (2). A
chatbot may offer `collections.Counter(...).most_common(1)` instead; same thing. Note
that `merges` records the merges in insertion order, which `encode` relies on.

In [ ]:
def train_bpe(text, vocab_size, base=256, verbose=False):
    """Learn BPE merges from `text`. Returns {pair: new_id} in the order learned."""
    ids = list(text.encode("utf-8"))
    merges = {}
    for i in range(vocab_size - base):
        stats = get_stats(ids)
        if not stats:
            break
        pair = max(stats, key=stats.get)     # most frequent pair
        idx = base + i
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        if verbose:
            print(f"merge {i+1}: {pair} -> {idx}")
    return merges

_m = train_bpe("the cat sat on the mat. " * 50, vocab_size=270)
print("train_bpe OK; learned", len(_m), "merges")

**Solution — `encode` (Section 1.5)**

Each pass applies the *earliest-learned* merge available: `merges.get(p, float("inf"))`
scores unknown pairs as infinity, so `min` picks the known pair with the lowest id;
once even the winner is unknown, nothing mergeable remains and we stop.

In [ ]:
def encode(text, merges):
    """Tokenise `text` using learned `merges` (apply earliest-learned merge first)."""
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # the pair with the LOWEST merge-id is the one learned earliest -> apply it first
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                      # nothing left that we know how to merge
        ids = merge(ids, pair, merges[pair])
    return ids

_m = train_bpe("banana banana banana ana ana", vocab_size=256 + 5)
print("round-trips:", decode(encode("bandana", _m), _m) == "bandana")

**Solution — encoding an unseen word (Section 1.6)**

`encode` only applies pairs it knows; anything unmerged stays as raw bytes, so an
unseen word can never fail to tokenise — the worst case is one token per byte.

In [ ]:
band_ids = encode("bandana", m_banana)
band_back = decode(band_ids, m_banana)
print("token ids  :", band_ids, f"({len(band_ids)} tokens for 7 bytes)")
print("round-trips:", band_back == "bandana")

**Solution — char-level vs BPE (Section 1.8)**

The character tokeniser is one token per character by construction, so
`len(char_ids) == len(sample)`. Expect roughly 2x compression with only 256 merges;
production tokenisers with ~100k-piece vocabularies do considerably better.

In [ ]:
sample = text[20000:20600]      # held-out text the tokeniser was not trained on
char_ids = P['encode'](sample)         # Lab-8 character tokeniser
bpe_ids  = encode(sample, merges)      # our BPE
compression = len(char_ids) / len(bpe_ids)

print(f"characters : {len(sample)}")
print(f"char tokens: {len(char_ids)}")
print(f"BPE tokens : {len(bpe_ids)}")
print(f"compression: {compression:.2f}x fewer tokens with BPE")
print("round-trips:", decode(bpe_ids, merges) == sample)

**Solution — the token-boundary prediction (Section 1.9)**

Far fewer than 15 tokens — `the` usually survives as one piece (often with its leading
space), while rarer words like `speaks` break up. The exact split depends on the merges
learned from the training slice: token boundaries are a property of the *corpus*, not
of English.

**Solution — the sampler (Section 2.1)**

Temperature: dividing by a small number stretches the gaps between logits, so softmax
gets peakier. Top-k: `v[:, [-1]]` is each row's k-th largest logit; everything below it
goes to `-inf`, which softmax turns into probability 0. Top-p: `cumprobs - sorted_probs`
is the cumulative probability *before* each token, so a token is removed only if the
nucleus was already complete without it — the smallest set that crosses `top_p`
survives.

In [ ]:
@torch.no_grad()
def generate_with_controls(model, idx, max_new_tokens, block_size,
                           temperature=1.0, top_k=None, top_p=None):
    """Autoregressively sample from `model` with temperature / top-k / top-p controls."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]               # crop context to block_size
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]                     # (B, vocab) -- last position only
        logits = logits / temperature                 # (1) temperature

        if top_k is not None:                         # (2) top-k
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('inf')

        if top_p is not None:                         # (3) top-p (nucleus)
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cumprobs = sorted_probs.cumsum(dim=-1)
            # remove tokens once the cumulative prob BEFORE them already exceeds top_p
            remove = (cumprobs - sorted_probs) > top_p
            sorted_logits[remove] = -float('inf')
            logits = logits.scatter(1, sorted_idx, sorted_logits)

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

print("generate_with_controls OK")

**Solution — greedy vs sampled (Section 2.4)**

At $\tau = 0.05$ almost all probability sits on one token, so the near-greedy pair
comes out (near-)identical. The $\tau = 1.0$ runs diverge within a few tokens, and
each divergence compounds — every token is conditioned on the ones before. Temperature
≈ 0 gives reproducible output: determinism, not truth.

In [ ]:
for temp in (0.05, 1.0):
    print(f"=== temperature {temp} ===")
    for run in (1, 2):
        out = generate_with_controls(gpt, start, 150, BLOCK_SIZE, temperature=temp)
        print(f"--- run {run} ---")
        print(P['decode'](out[0].tolist()))
    print()